# CONVERTING AQUIFER INFILTRATION RATE

Sanitised source snapshot. See the repository README and WORKFLOW.md before running. Outputs and execution metadata are removed. Historical experiments are not an unattended pipeline or reproduced scientific result.


In [ ]:
import os
from pathlib import Path

REPO_ROOT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "prepare_wells.py").is_file())
WORKSPACE_ROOT = Path(os.environ.get("CGWB_WORKSPACE", str(REPO_ROOT / "data" / "cgwb"))).expanduser().resolve()
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE_ROOT)


In [ ]:
import pandas as pd
import re

In [ ]:
# Modified input string from user
raw_data_mixed = """
Alluvium,AL01,Younger Alluvium(Clay/Silt/Sand/ Calcareousconcretions),Quaternary ,22,20,24
Alluvium,AL02,Pebble/ Gravel/ Bazada/Kandi,Quaternary,22,20,24
Alluvium,AL03,Older Alluvium (Silt/Sand/Gravel/Lithomargic clay),Quaternary,22,20,24
Alluvium,AL04,Aeolian Alluvium (Silt/ Sand),Quaternary,22,20,24
Alluvium,AL05,Coastal Alluvium (Sand/Silt/Clay) -East Coast,Quaternary,16,14,18
Alluvium,AL05,Coastal Alluvium (Sand/Silt/Clay) - West Coast,Quaternary,10,8,12
Alluvium,AL06,Valley Fills,Quaternary,22,20,24
Alluvium,AL07,Glacial Deposits,Quaternary,22,20,24
Laterite,LT01,Laterite/ Ferruginous concretions,Quaternary,7,6,8
Basalt,BS01,Basic Rocks (Basalt) - Vesicular or Jointed,Mesozoic to Cenozoic,13,12,14
Basalt,BS01,Basic Rocks (Basalt) - Weathered,Mesozoic to Cenozoic,7,6,8
Basalt,BS01,Basic Rocks (Basalt) - Massive Poorly Jointed,Mesozoic to Cenozoic,2,1,3
Basalt,BS02,Ultra Basic - Vesicular or Jointed,Mesozoic to Cenozoic,13,12,14
Basalt,BS02,Ultra Basic - Weathered,Mesozoic to Cenozoic,7,6,8
Basalt,BS02,Ultra Basic Jointed- Massive Poorly,Mesozoic to Cenozoic,2,1,3
"""

# Append unstructured lines and normalize them into CSV-like format
unstructured_lines = """
Sandstone ST01 Sandstone/Conglomerate Upper Palaeozoic to Cenozoic 12 10 14
Sandstone ST02 Sandstone with Shale Upper Palaeozoic to Cenozoic 12 10 14
Sandstone, ST03, Sandstone with shale/ coal beds,Upper Palaeozoic to Cenozoic, 12, 10, 14
Sandstone, ST04, Sandstone with Clay, Upper Palaeozoic to Cenozoic, 12, 10, 14
Sandstone, ST05, Sandstone/Conglomerate, Proterozoic to Cenozoic, 6, 5, 7
Sandstone, ST06, Sandstone with Shale, Proterozoic to Cenozoic, 6, 5, 7
Shale, SH01, Shale with limestone, Upper Palaeozoic to Cenozoic, 4, 3, 5
Shale, SH02, Shale with Sandstone, Upper Palaeozoic to Cenozoic, 4, 3, 5
Shale, SH03, Shale limestone and sandstone, Upper Palaeozoic to Cenozoic, 4, 3, 5
Shale, SH04, Shale, Upper Palaeozoic to Cenozoic, 4, 3, 5
Shale, SH05 Shale/Shale with Sandstone Proterozoic to Cenozoic 4 3 5
Shale SH06 Shale with Limestone Proterozoic to Cenozoic 4 3 5
Limestone LS01 Miliolitic Limestone Quarternary 6 5 7
Limestone LS02 Limestone/ Dolomite Upper Palaeozoic to Cenozoic 6 5 7
Limestone LS03 Limestone/Dolomite Proterozoic 6 5 7
Limestone LS04 Limestone with Shale Proterozoic 6 5 7
Limestone LS05 Marble Azoic to Proterozoic 6 5 7
Granite GR01 Acidic Rocks (Granite,Syenite, Rhyolite etc.) - Weathered , Jointed Mesozoic to Cenozoic 7 5 9
Granite GR01 Acidic Rocks (Granite,Syenite, Rhyolite etc.)-Massive or Poorly Fractured Mesozoic to Cenozoic 2 1 3
Granite GR02 Acidic Rocks (Pegmatite, Granite, Syenite, Rhyolite etc.) - Weathered, Jointed Proterozoic to Cenozoic 11 10 12
Granite GR02 Acidic Rocks (Pegmatite, Granite, Syenite, Rhyolite etc.) - Massive, Poorly Fractured Proterozoic to Cenozoic 2 1 3
Schist SC01 Schist - Weathered, Jointed Azoic to Proterozoic 7 5 9
Schist SC01 Schist - Massive, Poorly Fractured Azoic to Proterozoic 2 1 3
Schist SC02 Phyllite Azoic to Proterozoic 4 3 5
Schist SC03 Slate Azoic to Proterozoic 4 3 5
Quartzite QZ01 Quartzite - Weathered, Jointed Proterozoic to Cenozoic 6 5 7
Quartzite QZ01 Quartzite - Massive, Poorly Fractured Proterozoic to Cenozoic 2 1 3
Quartzite QZ02 Quartzite - Weathered, Jointed Azoic to Proterozoic 6 5 7
Quartzite QZ02 Quartzite- Massive, Poorly Fractured Azoic to Proterozoic 2 1 3
Charnockite CK01 Charnockite - Weathered, Jointed Azoic 5 4 6
Charnockite CK01 Charnockite - Massive, Poorly Fractured Azoic 2 1 3
Khondalite KH01 Khondalites, Granulites - Weathered, Jointed Azoic 7 5 9
Khondalite KH01 Khondalites, Granulites - Mssive, Poorly Fractured Azoic 2 1 3
Banded Gneissic Complex BG01 Banded Gneissic Complex - Weathered, Jointed Azoic 7 5 9
Banded Gneissic Complex BG01 Banded Gneissic Complex - Massive, Poorly Fractured Azoic 2 1 3
Gneiss GN01 Undifferentiated metasedimentaries/ Undifferentiated metamorphic - Weathered, Jointed Azoic to Proterozoic 7 5 9
Gneiss GN01 Undifferentiated metasedimentaries/ Undifferentiated metamorphic - Massive, Poorly Fractured Azoic to Proterozoic 2 1 3
Gneiss GN02 Gneiss -Weathered, Jointed Azoic to Proterozoic 11 10 12
Gneiss GN02 Gneiss-Massive, Poorly Fractured Azoic to Proterozoic 2 1 3
Gneiss GN03 Migmatitic Gneiss - Weathered, Jointed Azoic 7 5 9
Gneiss GN03 Migmatitic Gneiss - Massive, Poorly Fractured Azoic 2 1 3
Intrusive IN01 Basic Rocks (Dolerite, Anorthosite etc.) - Weathered, Jointed Proterozoic to Cenozoic 7 6 8
Intrusive IN01 Basic Rocks (Dolerite, Anorthosite etc.) - Massive, Poorly Fractured Proterozoic to Cenozoic 2 1 3
Intrusive IN02 Ulrta Basics (Epidiorite, Granophyre etc.) - Weathered, Jointed Proterozoic to Cenozoic 7 6 8
Intrusive IN02 Ulrta Basics (Epidiorite, Granophyre etc.) - Massive, Poorly Fractured Proterozoic to Cenozoic 2 1 3
"""


In [ ]:
unstructured_lines = re.sub(r'[^A-Za-z0-9\s]', ' ', unstructured_lines)

In [ ]:
import re
import pandas as pd
import re
import pandas as pd

# Define column headers
columns = ['Aquifer', 'Code', 'Detailed Aquifer Name', 'Geological Age', 'Recommended Rate', 'Minimum Rate', 'Maximum Rate']

# Function to parse a single unstructured line
def parse_line_with_flexible_age(line):
    parts = line.strip().split()
    if len(parts) < 7:
        return None

    # Detect code position (e.g., ST03, GR01, etc.)
    code_index = None
    for i, part in enumerate(parts):
        if re.fullmatch(r'[A-Za-z]+\d+', part):
            code_index = i
            break
    if code_index is None or code_index == 0:
        return None

    aquifer = " ".join(parts[:code_index])
    code = parts[code_index]

    try:
        rec_rate, min_rate, max_rate = int(parts[-3]), int(parts[-2]), int(parts[-1])
    except ValueError:
        return None

    # Identify geological age with optional 'to'
    if 'to' in parts[-6:-3]:
        to_idx = parts.index('to', -6, -3)
        geological_age = parts[to_idx - 1] + ' to ' + parts[to_idx + 1]
        detailed_name = " ".join(parts[code_index + 1:to_idx - 1])
    else:
        geological_age = parts[-4]
        detailed_name = " ".join(parts[code_index + 1:-4])

    return [aquifer.strip(), code.strip(), detailed_name.strip(), geological_age.strip(),
            rec_rate, min_rate, max_rate]

# Clean unstructured lines (remove punctuation, keep letters, digits, and spaces)
unstructured_clean = re.sub(r'[^A-Za-z0-9\s]', '', unstructured_lines)

# Apply parser to each cleaned unstructured line
structured_lines_cleaned = []
for line in unstructured_clean.strip().split('\n'):
    parsed = parse_line_with_flexible_age(line)
    if parsed:
        structured_lines_cleaned.append(','.join(map(str, parsed)))

# Combine with original structured lines (already comma-separated)
final_lines = raw_data_mixed.strip().split('\n') + structured_lines_cleaned

# Create DataFrame
df_combined = pd.DataFrame(
    [line.split(',') for line in final_lines if len(line.split(',')) == 7],
    columns=columns
)

# Strip whitespace
df_combined = df_combined.applymap(lambda x: x.strip() if isinstance(x, str) else x)

In [ ]:

malformed_lines = [line for line in final_lines if len(line.split(',')) != 7]

# Show a sample of these malformed lines with their column counts
malformed_info = [(line, len(line.split(','))) for line in malformed_lines]

df_malformed = pd.DataFrame(malformed_info, columns=["Line", "Column Count"])

In [ ]:
df_malformed

In [ ]:
df_combined

In [ ]:
df_combined['Aquifer'].value_counts()

In [ ]:
df_combined

In [ ]:
# Define both paths
path_shallow = f"{WORKSPACE_ROOT}/23. Training_shallow_five_years/input/infiltration_combined.csv"
path_deep = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/input/infiltration_combined.csv"

# Save to both locations
df_combined.to_csv(path_shallow, index=False)
df_combined.to_csv(path_deep, index=False)

print("Saved df_combined to both shallow and deep directories.")